# TTM × SIX G1 — Tiền kiểm Colab (Preflight)

Chỉ kiểm tra phiên bản, GPU, bộ nhớ và nguồn mã. **Không tải trọng số Qwen, không huấn luyện, không dùng dữ liệu nghiên cứu.**

Mở **Runtime → Change runtime type → T4 GPU** nếu có sẵn GPU. Chạy đúng một ô mã dưới đây. Nếu báo `HOLD_ENV_OR_GUARD`, giữ nguyên kết quả và gửi lại biên bản; **không sửa hoặc nâng cấp gói tùy tiện**. Một lượt smoke riêng chỉ được thực hiện sau khi tiền kiểm hợp lệ.

Script nguồn cố định theo commit `016609c3465c768e749b351e4f7ec4088f2a1a5a`, Git blob `24318de7d0196944a0a9a6d6be9851099f18b85b`.


In [ ]:
# TTM × SIX G1 — PRECHECK ONLY; không tải Qwen hoặc huấn luyện.
import hashlib, importlib.metadata as md, json, os, pathlib, platform
import shutil, subprocess, sys, urllib.request

PINNED_SCRIPT_COMMIT = "016609c3465c768e749b351e4f7ec4088f2a1a5a"
PINNED_BLOB = "24318de7d0196944a0a9a6d6be9851099f18b85b"
SCRIPT_PATH = "scripts/research/six_ttm/g1_qwen06_zero_science_smoke.py"
URL = f"https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/{PINNED_SCRIPT_COMMIT}/{SCRIPT_PATH}"
ROOT = pathlib.Path("/content/ttm_six_g1")
PACKAGES = ["torch", "transformers", "huggingface-hub", "tokenizers", "safetensors"]
print("G1_NOTEBOOK_MODE=ZERO_SCIENCE_PREFLIGHT_ONLY")
print("G1_NOTEBOOK_PYTHON=" + platform.python_version())
for pkg in PACKAGES:
    try: print("G1_PACKAGE_" + pkg + "=" + md.version(pkg))
    except md.PackageNotFoundError: print("G1_PACKAGE_" + pkg + "=NOT_INSTALLED")
try:
    import torch
    print("G1_CUDA_AVAILABLE=" + str(torch.cuda.is_available()))
    if torch.cuda.is_available():
        print("G1_GPU_NAME=" + torch.cuda.get_device_name(0))
        print("G1_GPU_CC=" + str(torch.cuda.get_device_capability(0)))
        free,total=torch.cuda.mem_get_info(0)
        print("G1_GPU_FREE_BYTES=" + str(free))
        print("G1_GPU_TOTAL_BYTES=" + str(total))
except Exception as exc:
    print("G1_GPU_INSPECTION_ERROR=" + str(exc)[:180])
if pathlib.Path("/content").is_dir():
    print("G1_DISK_FREE_BYTES=" + str(shutil.disk_usage("/content").free))

# Small exact-source download (~7 KB) only; NO teacher/model download.
with urllib.request.urlopen(URL, timeout=45) as response:
    src = response.read()
payload = b"blob " + str(len(src)).encode("ascii") + b"\0" + src
git_blob = hashlib.sha1(payload).hexdigest()
print("G1_SOURCE_BLOB_SHA=" + git_blob)
if git_blob != PINNED_BLOB:
    raise RuntimeError("STOP_G1_SOURCE_BLOB_MISMATCH_NO_EXECUTION")

if not pathlib.Path("/content").is_dir():
    raise RuntimeError("STOP_G1_NOT_COLAB")
ROOT.mkdir(parents=True, exist_ok=True)
local_script=ROOT/"g1_qwen06_zero_science_smoke.py"
local_script.write_bytes(src)
proc=subprocess.run([sys.executable, str(local_script), "--phase", "preflight"], check=False)
print("G1_PREFLIGHT_EXIT_CODE=" + str(proc.returncode))
receipt=ROOT/"G1_PREFLIGHT_RECEIPT.json"
if receipt.is_file():
    data=json.loads(receipt.read_text("utf-8"))
    print("G1_PREFLIGHT_RECEIPT=" + json.dumps(data, ensure_ascii=False, sort_keys=True))
else:
    print("G1_PREFLIGHT_RECEIPT=NOT_WRITTEN")
if proc.returncode:
    print("G1_FINAL=HOLD_ENV_OR_GUARD")
else:
    print("G1_FINAL=PASS_PREFLIGHT_ONLY_NO_SMOKE_NO_SCIENCE")
